# Unified RL-Assisted MPC Combined Supervisor

In [ ]:
from pathlib import Path
import os

RUN_MODE = "nominal"  # switch to "disturb" for disturbance mode
STYLE_PROFILE = "hybrid"  # one of: "hybrid", "paper", "debug"
SAVE_PDF = False

ENABLE_HORIZON = True
HORIZON_STATE_MODE = "standard"

ENABLE_MATRIX = True
MATRIX_AGENT_KIND = "td3"  # switch to "sac" for SAC runs
MATRIX_STATE_MODE = "standard"

ENABLE_WEIGHTS = True
WEIGHTS_AGENT_KIND = "td3"  # switch to "sac" for SAC runs
WEIGHTS_STATE_MODE = "standard"

ENABLE_RESIDUAL = True
RESIDUAL_AGENT_KIND = "td3"  # switch to "sac" for SAC runs
RESIDUAL_STATE_MODE = "standard"
USE_RHO_AUTHORITY = True  # only applies when residual mismatch mode is active


def resolve_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "Simulation").exists() and (candidate / "utils").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current notebook working directory.")


def build_combined_suffix() -> str:
    parts = []
    if ENABLE_HORIZON:
        parts.append(f"h_dqn_{HORIZON_STATE_MODE}")
    if ENABLE_MATRIX:
        parts.append(f"m_{MATRIX_AGENT_KIND}_{MATRIX_STATE_MODE}")
    if ENABLE_WEIGHTS:
        parts.append(f"w_{WEIGHTS_AGENT_KIND}_{WEIGHTS_STATE_MODE}")
    if ENABLE_RESIDUAL:
        residual_tag = f"r_{RESIDUAL_AGENT_KIND}_{RESIDUAL_STATE_MODE}"
        if RESIDUAL_STATE_MODE == "mismatch":
            residual_tag += "_rho" if USE_RHO_AUTHORITY else "_no_rho"
        parts.append(residual_tag)
    return "__".join(parts) if parts else "no_agents"


REPO_ROOT = resolve_repo_root(Path.cwd())
os.chdir(REPO_ROOT)

RUN_PROFILE_MAP = {
    "nominal": {
        "baseline_mpc_path": REPO_ROOT / "Data" / "mpc_results_nominal.pickle",
        "result_prefix_root": "combined_nominal",
        "compare_prefix_root": "nominal_compare_combined",
        "compare_mode": "nominal",
        "plot_start_episode": 2,
        "compare_start_episode": 2,
    },
    "disturb": {
        "baseline_mpc_path": REPO_ROOT / "Data" / "mpc_results_dist.pickle",
        "result_prefix_root": "combined_disturb",
        "compare_prefix_root": "disturb_compare_combined",
        "compare_mode": "disturb",
        "plot_start_episode": 2,
        "compare_start_episode": 2,
    },
}

for state_mode in (HORIZON_STATE_MODE, MATRIX_STATE_MODE, WEIGHTS_STATE_MODE, RESIDUAL_STATE_MODE):
    if state_mode not in {"standard", "mismatch"}:
        raise ValueError("All state modes must be 'standard' or 'mismatch'.")
for agent_kind in (MATRIX_AGENT_KIND, WEIGHTS_AGENT_KIND, RESIDUAL_AGENT_KIND):
    if agent_kind not in {"td3", "sac"}:
        raise ValueError("Continuous agent kinds must be 'td3' or 'sac'.")
if STYLE_PROFILE not in {"hybrid", "paper", "debug"}:
    raise ValueError("STYLE_PROFILE must be 'hybrid', 'paper', or 'debug'.")

RUN_PROFILE = RUN_PROFILE_MAP[RUN_MODE]
RUN_SUFFIX = build_combined_suffix()
RESULT_PREFIX = f"{RUN_PROFILE['result_prefix_root']}_{RUN_SUFFIX}"
COMPARE_PREFIX = f"{RUN_PROFILE['compare_prefix_root']}_{RUN_SUFFIX}"
print(f"Repo root: {REPO_ROOT}")
print(f"Run mode: {RUN_MODE} | Style profile: {STYLE_PROFILE}")
print(f"Combined suffix: {RUN_SUFFIX}")
print(f"Baseline MPC: {RUN_PROFILE['baseline_mpc_path']}")


In [ ]:
import numpy as np
import torch

from DQN.dqn_agent import DQNAgent
from SACAgent.sac_agent import SACAgent
from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from TD3Agent.agent import TD3Agent
from utils.combined_runner import run_combined_supervisor
from utils.helpers import apply_min_max, build_horizon_recipes, load_and_prepare_system_data
from utils.observer import compute_observer_gain
from utils.plotting import plot_combined_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim


In [ ]:
# First initiate the system
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
delta_t = 0.5
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {
    "ss_inputs": cstr_ss.ss_inputs,
    "y_ss": cstr_ss.y_ss,
}

setpoint_y = np.array([[3.2, 321.0], [4.5, 325.0]])
u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324.0], [3.4, 321.0]])
y_sp_scenario = (
    apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
    - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
)

n_tests = 200
set_points_len = 400
TEST_CYCLE = [False, False, False, False, False]
warm_start = 5
COMPARE_START_EPISODE = max(int(RUN_PROFILE.get("compare_start_episode", 1)), warm_start + 1)

poles = np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966, 0.42900673, 0.4228262, 0.96916776, 0.91230187])
observer_gain_preview = compute_observer_gain(A_aug, C_aug, poles)
print("Observer gain preview shape:", observer_gain_preview.shape)


In [ ]:
N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ACTOR_FREEZE = 0
DECISION_INTERVAL = 4

PREDICT_GRID = list(range(8, 20))
CONTROL_GRID = list(range(3, 10))
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
predict_h = 9
cont_h = 3

Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0
MODEL_LOW = np.array([0.95, 0.95, 0.95])
MODEL_HIGH = np.array([1.05, 1.05, 1.05])
WEIGHTS_LOW = np.array([0.9, 0.9, 0.9, 0.9])
WEIGHTS_HIGH = np.array([1.1, 1.1, 1.1, 1.1])
RESIDUAL_LOW = np.array([-0.5, -0.5])
RESIDUAL_HIGH = np.array([0.5, 0.5])

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty], float),
    R_in=np.array([R1_penalty, R2_penalty], float),
    NP=predict_h,
    NC=cont_h,
)

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])
Q_diag = np.array([518.0, 90.0])
R_diag = np.array([90.0, 90.0])
reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    N_INPUTS,
    k_rel,
    band_floor_phys,
    Q_diag,
    R_diag,
    tau_frac=0.7,
    gamma_out=0.5,
    gamma_in=0.5,
    beta=7.0,
    gate="geom",
    lam_in=1.0,
    bonus_kind="exp",
    bonus_k=12.0,
    bonus_p=0.6,
    bonus_c=20.0,
)
print(reward_params)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.85
qs_change = 1.3
ha_change = 0.85


def make_continuous_agent(agent_kind, state_dim, action_dim):
    if agent_kind == "td3":
        return TD3Agent(
            state_dim=state_dim,
            action_dim=action_dim,
            actor_hidden=[512, 512, 512],
            critic_hidden=[512, 512, 512],
            gamma=0.995,
            actor_lr=1e-4,
            critic_lr=1e-4,
            batch_size=128,
            policy_delay=4,
            target_policy_smoothing_noise_std=0.1,
            noise_clip=0.2,
            max_action=1.0,
            tau=0.005,
            std_start=0.2,
            std_end=0.02,
            std_decay_rate=0.99995,
            std_decay_mode="exp",
            buffer_size=25_000,
            device=DEVICE,
            actor_freeze=ACTOR_FREEZE,
        )
    if agent_kind == "sac":
        return SACAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            actor_hidden=[512, 512, 512],
            critic_hidden=[512, 512, 512],
            gamma=0.995,
            actor_lr=1e-4,
            critic_lr=1e-4,
            alpha_lr=1e-4,
            batch_size=128,
            grad_clip_norm=10.0,
            init_alpha=0.01,
            learn_alpha=True,
            target_entropy=-action_dim,
            target_update="soft",
            tau=0.005,
            hard_update_interval=10_000,
            activation="relu",
            use_layernorm=False,
            dropout=0.0,
            max_action=1.0,
            buffer_size=20_000,
            use_per=True,
            device=DEVICE,
            use_adamw=True,
            actor_freeze=ACTOR_FREEZE,
        )
    raise ValueError("Continuous agent kind must be 'td3' or 'sac'.")


horizon_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, HORIZON_STATE_MODE)
matrix_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, MATRIX_STATE_MODE)
weights_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, WEIGHTS_STATE_MODE)
residual_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, RESIDUAL_STATE_MODE)

horizon_agent = None
if ENABLE_HORIZON:
    horizon_agent = DQNAgent(
        state_dim=horizon_state_dim,
        action_dim=len(HORIZON_RECIPES),
        hidden_dim=[512, 512, 512, 512, 512],
        gamma=0.99,
        lr=1e-4,
        batch_size=128,
        buffer_size=40_000,
        grad_clip_norm=10.0,
        double_dqn=True,
        target_update="soft",
        tau=0.01,
        hard_update_interval=10_000,
        target_combine="q1",
        activation="relu",
        use_layer_norm=False,
        dropout=0.0,
        device=DEVICE,
        eps_start=0.3,
        eps_end=0.01,
        eps_decay_rate=0.99999,
        eps_decay_mode="exp",
    )

matrix_agent = make_continuous_agent(MATRIX_AGENT_KIND, matrix_state_dim, 1 + N_INPUTS) if ENABLE_MATRIX else None
weights_agent = make_continuous_agent(WEIGHTS_AGENT_KIND, weights_state_dim, 4) if ENABLE_WEIGHTS else None
residual_agent = make_continuous_agent(RESIDUAL_AGENT_KIND, residual_state_dim, N_INPUTS) if ENABLE_RESIDUAL else None

print(f"Using device: {DEVICE}")
print(f"Enabled agents: horizon={ENABLE_HORIZON}, matrix={ENABLE_MATRIX}, weights={ENABLE_WEIGHTS}, residual={ENABLE_RESIDUAL}")
print(f"State dimensions | horizon={horizon_state_dim}, matrix={matrix_state_dim}, weights={weights_state_dim}, residual={residual_state_dim}")


In [ ]:
combined_cfg = {
    "run_mode": RUN_MODE,
    "decision_interval": DECISION_INTERVAL,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "horizon_cfg": {
        "enabled": ENABLE_HORIZON,
        "state_mode": HORIZON_STATE_MODE,
        "horizon_recipes": HORIZON_RECIPES,
        "default_horizons": (predict_h, cont_h),
    },
    "matrix_cfg": {
        "enabled": ENABLE_MATRIX,
        "agent_kind": MATRIX_AGENT_KIND,
        "state_mode": MATRIX_STATE_MODE,
        "low_coef": MODEL_LOW,
        "high_coef": MODEL_HIGH,
    },
    "weight_cfg": {
        "enabled": ENABLE_WEIGHTS,
        "agent_kind": WEIGHTS_AGENT_KIND,
        "state_mode": WEIGHTS_STATE_MODE,
        "low_coef": WEIGHTS_LOW,
        "high_coef": WEIGHTS_HIGH,
    },
    "residual_cfg": {
        "enabled": ENABLE_RESIDUAL,
        "agent_kind": RESIDUAL_AGENT_KIND,
        "state_mode": RESIDUAL_STATE_MODE,
        "use_rho_authority": USE_RHO_AUTHORITY,
        "low_coef": RESIDUAL_LOW,
        "high_coef": RESIDUAL_HIGH,
    },
}

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
agents = {}
if ENABLE_HORIZON:
    agents["horizon"] = horizon_agent
if ENABLE_MATRIX:
    agents["matrix"] = matrix_agent
if ENABLE_WEIGHTS:
    agents["weights"] = weights_agent
if ENABLE_RESIDUAL:
    agents["residual"] = residual_agent

runtime_ctx = {
    "system": cstr,
    "agents": agents,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "reward_params": reward_params,
}

result_bundle = run_combined_supervisor(combined_cfg=combined_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = RUN_PROFILE["baseline_mpc_path"]


In [ ]:
out_dir_rl = plot_combined_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": REPO_ROOT / "Data",
        "prefix_name": RESULT_PREFIX,
        "start_episode": RUN_PROFILE["plot_start_episode"],
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
        "reward_fn": reward_fn,
        "include_baseline_compare": True,
        "compare_mode": RUN_PROFILE["compare_mode"],
        "compare_prefix": COMPARE_PREFIX,
        "compare_directory": REPO_ROOT / "Result",
        "mpc_path_or_dir": RUN_PROFILE["baseline_mpc_path"],
        "compare_start_episode": COMPARE_START_EPISODE,
    },
)

print("Combined result directory:", out_dir_rl)
